# No-comms MVP (paper-faithful): geometry + behavior + von Mises

This notebook extends the no-comms MVP to include **paper-faithful behavioral metrics** and strict **von Mises-based interference**.

## Strict rules in this notebook
- Include expected no-comms grid runs for **two-module** (`task_routed`, `shared`) and **capacity-matched single-module** baselines.
- **Do not skip missing files**: the notebook raises on missing simulation folders, missing `sim_*.npz`, missing `probes`, or missing von Mises CSV/rows.
- Use only paper-faithful metrics:
  - `transfer_error_diff = mean(first 6 B winter) - mean(last 6 A1 winter)`
  - `interference = 1 - A_weight_A2` (from von Mises fits)
  - `summer_accuracy` from A1 summer trials
  - `generalisation_acc` from A1 winter test-stim trials (ANN convention used in this repo)

Outputs are written to:
- `data/derived/no_comms_mvp_pca_long.csv`
- `data/derived/no_comms_mvp_angles_long.csv`
- `data/derived/no_comms_mvp_behavior_long.csv`
- `data/derived/no_comms_mvp_merged_long.csv`


In [1]:
from pathlib import Path
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

project_root = Path.cwd().resolve()
if project_root.name != "a1b2_modular":
    if (project_root / "a1b2_modular").exists():
        project_root = project_root / "a1b2_modular"
    else:
        for parent in [project_root, *project_root.parents]:
            if parent.name == "a1b2_modular":
                project_root = parent
                break
os.chdir(project_root)

from a1b2.utils.run_config import build_run_id
from a1b2.analysis import transfer_interference as ann

sim_root = project_root / "data" / "simulations"
derived = project_root / "data" / "derived"
derived.mkdir(parents=True, exist_ok=True)
config_path = project_root / "a1b2" / "models" / "experiments.json"
settings = json.loads(config_path.read_text())

INIT_ORDER = [0.001, 0.01, 0.1, 1.0, 2.0]
DIM_ORDER = [6, 12, 25, 50]
ROUTING_ORDER = ["task_routed", "shared", "single_module"]
BASELINE_SINGLE_HIDDEN = {6: 12, 12: 25, 25: 50, 50: 100}
SIM_ORDER = ["same", "near", "far"]
PHASE_ORDER = ["post_A", "post_B", "post_A2"]

try:
    import statsmodels.formula.api as smf
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print('statsmodels not installed — MixedLM section will be skipped. Install: pip install -e ".[notebook-stats]"')


/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def sparsity_label(c: dict) -> str:
    sp = c.get("sparsity", 1.0)
    if sp == 0 or "no_comms" in c.get("name", ""):
        return "no_comms"
    sf = float(sp)
    return "1.0" if abs(sf - 1.0) < 1e-9 else str(sf)


def init_scale_from_condition(c: dict) -> float:
    return float(1.0 if c.get("init_scale") is None else c.get("init_scale"))


def collect_expected_runs() -> pd.DataFrame:
    """Expected grid rows for strict coverage (no-comms two-module + single baseline)."""
    rows = []
    paper_inits = set(INIT_ORDER)

    # Two-module no_comms rows
    for grid_h in DIM_ORDER:
        for routing in ("task_routed", "shared"):
            for c in settings["conditions"]:
                if c.get("arch") != "two_module_rnn":
                    continue
                if c.get("dim_hidden") != grid_h:
                    continue
                if c.get("nb_steps", 1) != 2:
                    continue
                if c.get("common_readout", True) is not True:
                    continue
                if c.get("common_input", False) is not False:
                    continue
                if c.get("input_routing", "shared") != routing:
                    continue
                if c.get("init_scope") == "input_only":
                    continue
                if sparsity_label(c) != "no_comms":
                    continue
                ini = init_scale_from_condition(c)
                if ini not in paper_inits:
                    continue
                rid = build_run_id(c)
                folder = sim_root / rid
                rows.append(
                    {
                        "condition_name": c["name"],
                        "run_id": rid,
                        "routing": routing,
                        "grid_h": grid_h,
                        "dim_hidden": grid_h,
                        "model_family": "two_module",
                        "init_scale": ini,
                        "folder_path": str(folder),
                    }
                )

    # Capacity-matched single-module rows
    for grid_h in DIM_ORDER:
        sh = BASELINE_SINGLE_HIDDEN[grid_h]
        for c in settings["conditions"]:
            if c.get("arch") != "single_module_rnn":
                continue
            if c.get("dim_hidden") != sh:
                continue
            if c.get("n_modules", 1) != 1:
                continue
            if c.get("nb_steps", 1) != 2:
                continue
            if c.get("common_readout", True) is not True:
                continue
            if c.get("common_input", False) is not False:
                continue
            if abs(float(c.get("sparsity", 1.0)) - 1.0) > 1e-9:
                continue
            if c.get("init_scope") == "input_only":
                continue
            ini = init_scale_from_condition(c)
            if ini not in paper_inits:
                continue
            rid = build_run_id(c)
            folder = sim_root / rid
            rows.append(
                {
                    "condition_name": c["name"],
                    "run_id": rid,
                    "routing": "single_module",
                    "grid_h": grid_h,
                    "dim_hidden": sh,
                    "model_family": "single_module",
                    "init_scale": ini,
                    "folder_path": str(folder),
                }
            )

    out = pd.DataFrame(rows)
    out = out.sort_values(["grid_h", "routing", "init_scale", "condition_name"], kind="stable").reset_index(drop=True)
    return out


def strict_validate_run_files(run_row: pd.Series) -> tuple[list[Path], Path]:
    """Raise if expected files are missing. Returns (npz_paths, vm_csv_path)."""
    folder = Path(run_row["folder_path"])
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing simulation folder: {folder}")
    npz_paths = sorted(folder.glob("sim_*.npz"))
    if len(npz_paths) == 0:
        raise FileNotFoundError(f"No sim_*.npz in folder: {folder}")

    vm_csv = sim_root / f"{run_row['run_id']}_vonmises_fits.csv"
    if not vm_csv.is_file():
        raise FileNotFoundError(
            f"Missing von Mises CSV for run_id={run_row['run_id']}: {vm_csv}\n"
            f"Run: python3 scripts/03_fit_vonmises.py simulations --sim-name {run_row['run_id']} --base-folder ."
        )
    return npz_paths, vm_csv


runs_df = collect_expected_runs()
runs_df["path_exists"] = runs_df["folder_path"].map(lambda p: Path(p).is_dir())
runs_df["n_npz"] = runs_df["folder_path"].map(lambda p: len(list(Path(p).glob("sim_*.npz"))) if Path(p).is_dir() else 0)
display(runs_df)
print("Expected rows:", len(runs_df))


,condition_name,run_id,routing,grid_h,dim_hidden,model_family,init_scale,folder_path,path_exists,n_npz
0,two_module_rnn_6_no_comms_nb2_init0.001,two_module_rnn_6_no_comms_nb2_init0.001_nb2_sh...,shared,6,6,two_module,0.001,/home/kat/workspace/Structure-Function-Analysi...,True,305
1,two_module_rnn_6_no_comms_nb2_init0.01,two_module_rnn_6_no_comms_nb2_init0.01_nb2_sha...,shared,6,6,two_module,0.010,/home/kat/workspace/Structure-Function-Analysi...,True,305
2,two_module_rnn_6_no_comms_nb2_init0.1,two_module_rnn_6_no_comms_nb2_init0.1_nb2_shar...,shared,6,6,two_module,0.100,/home/kat/workspace/Structure-Function-Analysi...,True,305
3,two_module_rnn_6_no_comms_nb2,two_module_rnn_6_no_comms_nb2_nb2_shared_sp0_s...,shared,6,6,two_module,1.000,/home/kat/workspace/Structure-Function-Analysi...,True,305
4,two_module_rnn_6_no_comms_nb2_init2,two_module_rnn_6_no_comms_nb2_init2_nb2_shared...,shared,6,6,two_module,2.000,/home/kat/workspace/Structure-Function-Analysi...,True,305
...,...,...,...,...,...,...,...,...,...,...
81,two_module_rnn_50_task_routed_no_comms_nb2_ini...,two_module_rnn_50_task_routed_no_comms_nb2_ini...,task_routed,50,50,two_module,0.001,/home/kat/workspace/Structure-Function-Analysi...,True,305
82,two_module_rnn_50_task_routed_no_comms_nb2_ini...,two_module_rnn_50_task_routed_no_comms_nb2_ini...,task_routed,50,50,two_module,0.010,/home/kat/workspace/Structure-Function-Analysi...,True,305
83,two_module_rnn_50_task_routed_no_comms_nb2_ini...,two_module_rnn_50_task_routed_no_comms_nb2_ini...,task_routed,50,50,two_module,0.100,/home/kat/workspace/Structure-Function-Analysi...,True,305
84,two_module_rnn_50_task_routed_no_comms_nb2,two_module_rnn_50_task_routed_no_comms_nb2_nb2...,task_routed,50,50,two_module,1.000,/home/kat/workspace/Structure-Function-Analysi...,True,305


Expected rows: 86


In [4]:
# Preflight: enumerate missing von Mises files and print exact commands.
# This cell does not skip failures; it helps you fill gaps before the strict run.

missing_vm = []
for _, r in runs_df.iterrows():
    vm_csv = sim_root / f"{r['run_id']}_vonmises_fits.csv"
    if not vm_csv.is_file():
        missing_vm.append((r['run_id'], str(vm_csv)))

if not missing_vm:
    print('Preflight OK: all run-level von Mises CSV files are present.')
else:
    print(f'Missing von Mises CSVs: {len(missing_vm)}')
    for run_id, vm_path in missing_vm:
        print(f'- {vm_path}')

    print('Run these commands from a1b2_modular to generate missing files:')
    for run_id, _ in missing_vm:
        print(f'python3 scripts/03_fit_vonmises.py simulations --sim-name {run_id} --base-folder .')


Missing von Mises CSVs: 57
- /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_6_no_comms_nb2_init0.01_nb2_shared_sp0_sep_cr_RNN_init0.01_vonmises_fits.csv
- /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/single_module_rnn_12_nb2_init0.1_nb2_shared_sp1_sep_cr_RNN_init0.1_vonmises_fits.csv
- /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_6_task_routed_no_comms_nb2_init0.001_nb2_task_routed_sp0_sep_cr_RNN_init0.001_vonmises_fits.csv
- /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_6_task_routed_no_comms_nb2_init0.01_nb2_task_routed_sp0_sep_cr_RNN_init0.01_vonmises_fits.csv
- /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_6_task_routed_no_comms_nb2_init0.1_nb2_task_routed_

In [5]:
# Strict coverage check (no skip policy)
for _, r in runs_df.iterrows():
    _npz_paths, _vm_csv = strict_validate_run_files(r)

print("Strict file coverage check passed for all runs.")


FileNotFoundError: Missing von Mises CSV for run_id=two_module_rnn_6_no_comms_nb2_init0.01_nb2_shared_sp0_sep_cr_RNN_init0.01: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_6_no_comms_nb2_init0.01_nb2_shared_sp0_sep_cr_RNN_init0.01_vonmises_fits.csv
Run: python3 scripts/03_fit_vonmises.py simulations --sim-name two_module_rnn_6_no_comms_nb2_init0.01_nb2_shared_sp0_sep_cr_RNN_init0.01 --base-folder .

In [6]:
def collapse_by_probes(values_1d: np.ndarray, probes_1d: np.ndarray, probe_value: int) -> np.ndarray:
    values_1d = np.asarray(values_1d, dtype=float)
    probes_1d = np.asarray(probes_1d)
    mask = np.isfinite(values_1d) & np.isfinite(probes_1d) & (probes_1d == probe_value)
    return values_1d[mask]


def compute_behavior_metrics_strict(entry: dict) -> dict:
    """Paper-faithful ANN-side metrics from one participant entry. Raises on missing required keys."""
    if "probes" not in entry:
        raise KeyError(f"Missing 'probes' in participant entry {entry.get('participant')} — cannot do probe-faithful metrics.")

    acc = np.asarray(entry["accuracy"], dtype=float)
    probes = np.asarray(entry["probes"])  # same shape as acc expected
    test_stim = np.asarray(entry["test_stim"]) if "test_stim" in entry else None

    if acc.ndim != 2 or probes.ndim != 2:
        raise ValueError(f"accuracy/probes must be 2D; got {acc.shape}, {probes.shape}")
    if acc.shape != probes.shape:
        raise ValueError(f"accuracy/probes shape mismatch: {acc.shape} vs {probes.shape}")

    # Phase indices: 0=A1, 1=B, 2=A2
    a1_w = collapse_by_probes(acc[0], probes[0], probe_value=1)
    b_w = collapse_by_probes(acc[1], probes[1], probe_value=1)
    a2_w = collapse_by_probes(acc[2], probes[2], probe_value=1) if acc.shape[0] > 2 else np.array([], dtype=float)
    a1_s = collapse_by_probes(acc[0], probes[0], probe_value=0)

    if len(a1_w) < 6 or len(b_w) < 6:
        raise ValueError(
            f"Insufficient winter points for transfer metric for participant {entry.get('participant')}: "
            f"len(A1_w)={len(a1_w)}, len(B_w)={len(b_w)}"
        )

    transfer_error_diff = float(np.mean(b_w[:6]) - np.mean(a1_w[-6:]))
    summer_accuracy = float(np.mean(a1_s)) if len(a1_s) else np.nan
    retest_error_diff = float(np.mean(a2_w) - np.mean(a1_w[-6:])) if len(a2_w) else np.nan

    # ANN convention available in repo: A1 winter on test-stim trials
    if test_stim is None:
        raise KeyError(f"Missing 'test_stim' in participant entry {entry.get('participant')} — cannot compute generalisation_acc.")
    if test_stim.ndim != 2 or test_stim.shape != acc.shape:
        raise ValueError(f"test_stim shape mismatch: {test_stim.shape} vs accuracy {acc.shape}")

    a1_w_mask = np.isfinite(acc[0]) & np.isfinite(probes[0]) & (probes[0] == 1) & (test_stim[0].astype(int) == 1)
    gen_vals = acc[0][a1_w_mask]
    generalisation_acc = float(np.nanmean(gen_vals)) if len(gen_vals) else np.nan

    return {
        "transfer_error_diff": transfer_error_diff,
        "summer_accuracy": summer_accuracy,
        "generalisation_acc": generalisation_acc,
        "retest_error_diff": retest_error_diff,
    }


def load_vm_csv_strict(vm_csv: Path) -> pd.DataFrame:
    vm = pd.read_csv(vm_csv)
    needed = {"participant", "condition", "A_weight_A2"}
    missing_cols = needed.difference(vm.columns)
    if missing_cols:
        raise KeyError(f"von Mises CSV missing columns {sorted(missing_cols)}: {vm_csv}")
    if vm["A_weight_A2"].isna().any():
        bad = vm.loc[vm["A_weight_A2"].isna(), ["participant", "condition"]]
        raise ValueError(f"von Mises CSV has NaN A_weight_A2 rows in {vm_csv}:\n{bad.head()}\n...")
    return vm


In [7]:
pca_frames = []
angle_frames = []
behav_rows = []

for _, r in tqdm(list(runs_df.iterrows()), desc="Runs (strict)"):
    npz_paths, vm_csv = strict_validate_run_files(r)
    vm = load_vm_csv_strict(vm_csv)

    # Load ANN data (strict: no try/except skip)
    data = ann.load_ann_data(r["folder_path"], load_rnn_extra=False)

    # Strict participant coverage: vm rows must exist for every near/far participant in npz
    # (same may be absent in vm fit pipeline; interference is defined on near/far)
    near_far_participants = []
    for sim_name in ["near", "far"]:
        for entry in data.get(sim_name, []):
            near_far_participants.append((str(entry["participant"]), sim_name))

    vm_pairs = set(zip(vm["participant"].astype(str), vm["condition"].astype(str)))
    missing_vm_pairs = [(p, s) for (p, s) in near_far_participants if (p, s) not in vm_pairs]
    if missing_vm_pairs:
        raise RuntimeError(
            f"Missing von Mises rows for run_id={r['run_id']} (participant,condition): {missing_vm_pairs[:10]}"
        )

    # Geometry tables
    pca_part = ann.compute_pca_representation_metrics(
        data,
        variance_thresholds=(0.9, 0.99),
        top_k=2,
        include_paths=("combined",),
    )
    if pca_part.empty:
        raise RuntimeError(f"Empty PCA metrics for run {r['run_id']}")
    pca_part = pca_part.rename(columns={"condition": "similarity"})
    pca_part["run_id"] = r["run_id"]
    pca_part["condition_name"] = r["condition_name"]
    pca_part["routing"] = r["routing"]
    pca_part["grid_h"] = r["grid_h"]
    pca_part["dim_hidden"] = r["dim_hidden"]
    pca_part["model_family"] = r["model_family"]
    pca_part["init_scale"] = r["init_scale"]
    pca_part["log10_init"] = np.log10(pca_part["init_scale"].astype(float))
    pca_part["participant_run"] = pca_part["run_id"].astype(str) + "__" + pca_part["participant"].astype(str)
    pca_frames.append(pca_part)

    ang = ann.get_principal_angles(data).rename(columns={"condition": "similarity"})
    if ang.empty:
        raise RuntimeError(f"Empty principal-angle table for run {r['run_id']}")
    ang["run_id"] = r["run_id"]
    ang["condition_name"] = r["condition_name"]
    ang["routing"] = r["routing"]
    ang["grid_h"] = r["grid_h"]
    ang["dim_hidden"] = r["dim_hidden"]
    ang["model_family"] = r["model_family"]
    ang["init_scale"] = r["init_scale"]
    ang["log10_init"] = np.log10(ang["init_scale"].astype(float))
    ang["participant_run"] = ang["run_id"].astype(str) + "__" + ang["participant"].astype(str)
    angle_frames.append(ang)

    # Behavioral metrics + strict VM interference merge
    for sim_name in SIM_ORDER:
        for entry in data.get(sim_name, []):
            participant = str(entry["participant"])
            metrics = compute_behavior_metrics_strict(entry)
            row = {
                "participant": participant,
                "similarity": sim_name,
                "run_id": r["run_id"],
                "condition_name": r["condition_name"],
                "routing": r["routing"],
                "grid_h": r["grid_h"],
                "dim_hidden": r["dim_hidden"],
                "model_family": r["model_family"],
                "init_scale": r["init_scale"],
                "log10_init": float(np.log10(r["init_scale"])),
                "participant_run": f"{r['run_id']}__{participant}",
                **metrics,
            }

            # Interference: required from vm for near/far; same -> NaN by definition in this fit setup
            if sim_name in ("near", "far"):
                vm_match = vm[(vm["participant"].astype(str) == participant) & (vm["condition"].astype(str) == sim_name)]
                if len(vm_match) != 1:
                    raise RuntimeError(
                        f"Expected exactly one vm row for run={r['run_id']}, participant={participant}, condition={sim_name}; got {len(vm_match)}"
                    )
                row["interference"] = float(1.0 - vm_match.iloc[0]["A_weight_A2"])
            else:
                row["interference"] = np.nan

            behav_rows.append(row)

pca_long = pd.concat(pca_frames, ignore_index=True)
angles_long = pd.concat(angle_frames, ignore_index=True)
behavior_long = pd.DataFrame(behav_rows)

# Merge behavior with selected geometry snapshots for integrated analyses
pca_postb = pca_long[(pca_long["phase"] == "post_B") & (pca_long["pathway"] == "combined")].copy()
pca_postb = pca_postb[[
    "participant", "similarity", "run_id", "participant_run", "n_pcs_90", "n_pcs_99", "var_topk"
]].drop_duplicates()

angles_sel = angles_long[[
    "participant", "similarity", "run_id", "participant_run", "principal_angle_between"
]].drop_duplicates()

merged = behavior_long.merge(pca_postb, on=["participant", "similarity", "run_id", "participant_run"], how="left", validate="one_to_one")
merged = merged.merge(angles_sel, on=["participant", "similarity", "run_id", "participant_run"], how="left", validate="one_to_one")

# Strict: merged rows must have geometry populated
if merged[["n_pcs_99", "principal_angle_between"]].isna().any().any():
    bad = merged[merged[["n_pcs_99", "principal_angle_between"]].isna().any(axis=1)].head()
    raise RuntimeError(f"Merged table has missing geometry fields. Example rows:\n{bad}")

pca_path = derived / "no_comms_mvp_pca_long.csv"
ang_path = derived / "no_comms_mvp_angles_long.csv"
beh_path = derived / "no_comms_mvp_behavior_long.csv"
merged_path = derived / "no_comms_mvp_merged_long.csv"

pca_long.to_csv(pca_path, index=False)
angles_long.to_csv(ang_path, index=False)
behavior_long.to_csv(beh_path, index=False)
merged.to_csv(merged_path, index=False)

print("Wrote", pca_path, pca_long.shape)
print("Wrote", ang_path, angles_long.shape)
print("Wrote", beh_path, behavior_long.shape)
print("Wrote", merged_path, merged.shape)


Runs (strict):   1%|          | 1/86 [00:03<04:34,  3.23s/it]


FileNotFoundError: Missing von Mises CSV for run_id=two_module_rnn_6_no_comms_nb2_init0.01_nb2_shared_sp0_sep_cr_RNN_init0.01: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_6_no_comms_nb2_init0.01_nb2_shared_sp0_sep_cr_RNN_init0.01_vonmises_fits.csv
Run: python3 scripts/03_fit_vonmises.py simulations --sim-name two_module_rnn_6_no_comms_nb2_init0.01_nb2_shared_sp0_sep_cr_RNN_init0.01 --base-folder .

In [ ]:
def heatmap_by_routing_similarity(df, value_col, title, cmap="viridis", fmt=".2f", vmin=None, vmax=None):
    fig, axes = plt.subplots(3, 3, figsize=(12, 9.5), constrained_layout=True)
    for ri, routing in enumerate(ROUTING_ORDER):
        for ci, sim in enumerate(SIM_ORDER):
            ax = axes[ri, ci]
            chunk = df[(df["routing"] == routing) & (df["similarity"] == sim)]
            pt = chunk.pivot_table(index="grid_h", columns="init_scale", values=value_col, aggfunc="mean")
            pt = pt.reindex(index=DIM_ORDER, columns=INIT_ORDER)
            sns.heatmap(
                pt,
                ax=ax,
                annot=True,
                fmt=fmt,
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                cbar=(ri == 2 and ci == 2),
            )
            ax.set_title(f"{routing} | {sim}")
            ax.set_xlabel("init_scale")
            ax.set_ylabel("grid_h")
    fig.suptitle(title)
    plt.show()


# Geometry heatmaps
heatmap_by_routing_similarity(
    merged,
    value_col="n_pcs_99",
    title="Mean n_pcs_99 (post_B, combined pathway)",
    cmap="viridis",
    fmt=".1f",
)

heatmap_by_routing_similarity(
    merged,
    value_col="principal_angle_between",
    title="Mean principal angle (A vs B subspaces)",
    cmap="magma",
    fmt=".1f",
)

# Paper-faithful behavior heatmaps
heatmap_by_routing_similarity(
    merged,
    value_col="transfer_error_diff",
    title="Transfer (first 6 B winter - last 6 A1 winter)",
    cmap="coolwarm",
    fmt=".3f",
)

heatmap_by_routing_similarity(
    merged,
    value_col="interference",
    title="Interference = 1 - A_weight_A2 (von Mises)",
    cmap="rocket_r",
    fmt=".3f",
)

heatmap_by_routing_similarity(
    merged,
    value_col="generalisation_acc",
    title="Generalisation accuracy (A1 winter test-stim)",
    cmap="YlGnBu",
    fmt=".3f",
    vmin=0,
    vmax=1,
)


In [ ]:
# Mixed model on a paper-faithful primary behavioral target
# (interference uses von Mises by construction, near/far only)
if not HAS_STATSMODELS:
    print("statsmodels not installed; skipping MixedLM block.")
else:
    sub = merged[merged["similarity"].isin(["near", "far"])].copy()
    sub = sub.dropna(subset=["interference"])
    sub["similarity"] = pd.Categorical(sub["similarity"], categories=["near", "far"])
    sub["grid_h"] = pd.Categorical(sub["grid_h"], categories=DIM_ORDER)
    sub["routing"] = pd.Categorical(sub["routing"], categories=ROUTING_ORDER)

    # Random intercept at participant_run level (run-specific participant instance)
    model = smf.mixedlm(
        "interference ~ log10_init * C(similarity) + C(grid_h) + C(routing)",
        data=sub,
        groups=sub["participant_run"],
    )
    fit = model.fit(method="lbfgs", maxiter=200)
    print(fit.summary())
